# **Purpose**

This Notebook Tests and Evaluates the Fine Tuned TinyLlama 1.1B Model for MCQ Question Answer Generation

## **Install the required modules**

In [2]:
!pip install -qU \
evaluate\
nltk\
transformers\
torch\
datasets\
spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


**Restart the session after all the modules are installed.**

##**Import the required Libraries**

In [ ]:
!python -m spacy download en_core_web_sm

In [1]:
# Import required libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from datasets import load_dataset
from google.colab import drive
import evaluate

##**Initialize the Model from Google Drive**

In [4]:
# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Path where your merged model is saved on Google Drive
merged_model_path = "/content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq"

print("Loading tokenizer and fine-tuned merged model from Google Drive...")
tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
model = AutoModelForCausalLM.from_pretrained(
    merged_model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

Mounted at /content/drive
Loading tokenizer and fine-tuned merged model from Google Drive...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

###**Initialize the text-generation pipeline**

In [5]:

qg_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=128,
    do_sample=False  # Deterministic generation for evaluation benchmarks
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


###**Interactive Testing Function**

In [6]:
def generate_mcq_question(context, target_answer):
    messages = [
        {
            "role": "system",
            "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
        },
        {
            "role": "user",
            "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = qg_pipeline(prompt)
    return response[0]['generated_text']

####**Testing with First Sample**

In [7]:
test_context = "The Transformer architecture relies entirely on attention mechanisms to draw global dependencies between input and output, eschewing recurrence and convolutions entirely."
test_answer = "attention mechanisms"

print(generate_mcq_question(test_context, test_answer))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question: What is the Transformer architecture's primary mechanism for drawing global dependencies between input and output?
Answer: attention mechanisms


####**Testing with Second Sample**

In [8]:
# Test sample 2
test_context = "Electromagnetic induction is the production of an electromotive force across an electrical conductor in a changing magnetic field. Faraday's law of induction predicts how a magnetic field will interact with an electric circuit to produce an EMF."
test_answer = "Faraday's law"

print(generate_mcq_question(test_context, test_answer))

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What law predicts how a magnetic field will interact with an electric circuit to produce an EMF?
Answer: Faraday's law


##**Evaluation**

In [9]:
# Load SQuAD v2 validation split (taking a subset for quick benchmarking, e.g., 200 samples)
val_dataset = load_dataset('rajpurkar/squad_v2', split='validation').shuffle(seed=42).select(range(1500))

references = []
predictions = []

print("Running inference on SQuAD v2 validation split...")
for example in val_dataset:
    context = example["context"]
    answers = example.get("answers", {})

    # Skip unanswerable questions if target answer text is empty
    if not answers or len(answers.get("text", [])) == 0:
        continue

    target_answer = answers["text"][0]
    true_question = example["question"]

    messages = [
        {
            "role": "system",
            "content": "You are an expert educational assessment AI that generates a clear, high-quality question based strictly on a given context and target answer."
        },
        {
            "role": "user",
            "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
        }
    ]

    # Format prompt using the chat template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Generate output
    output = qg_pipeline(prompt)
    generated_text = output[0]['generated_text']

    # Simple parsing to extract the question text
    pred_question = generated_text.split("Answer:")[0].replace("Question:", "").strip()

    predictions.append(pred_question)
    references.append(true_question)


README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running inference on SQuAD v2 validation split...


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

##**Compute BLEU and ROUGE scores**

In [11]:
# Load evaluation metrics specified in the project report (BLEU and ROUGE)
!pip install rouge_score
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

# Compute metrics
bleu_results = bleu_metric.compute(predictions=predictions, references=references)
rouge_results = rouge_metric.compute(predictions=predictions, references=references)

print("\n--- Evaluation Benchmarks on SQuAD v2 Validation Split ---")
print(f"BLEU Score: {bleu_results['bleu']:.4f}")
print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=cda348491b8b7a1539349a1cc50e1058cc9b7326ae2008f1e5afe4faf27fd606
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score



--- Evaluation Benchmarks on SQuAD v2 Validation Split ---
BLEU Score: 0.1359
ROUGE-1: 0.4077
ROUGE-2: 0.1840
ROUGE-L: 0.3717


In [12]:
!pip install -q evaluate nltk transformers torch

In [13]:
import nltk
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [14]:
import math
import torch
import evaluate
from collections import Counter
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# -------------------------------------------------------------
# 1. METEOR Metric
# -------------------------------------------------------------
meteor_metric = evaluate.load("meteor")
# Usage: predictions and references lists from your validation loop
meteor_results = meteor_metric.compute(predictions=predictions, references=references)

# -------------------------------------------------------------
# 2. Token-level F1-Score Metric
# -------------------------------------------------------------
def compute_token_f1(pred_str, ref_str):
    pred_tokens = pred_str.lower().split()
    ref_tokens = ref_str.lower().split()

    if not pred_tokens or not ref_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(ref_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

avg_f1_score = sum(compute_token_f1(p, r) for p, r in zip(predictions, references)) / len(predictions)

# -------------------------------------------------------------
# 3. Perplexity (Evaluated using standard GPT-2)
# -------------------------------------------------------------
def compute_perplexity(texts, model_id="gpt2"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ppl_tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
    ppl_model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
    ppl_model.eval()

    nlls = []
    for text in texts:
        if not text.strip():
            continue
        encodings = ppl_tokenizer(text, return_tensors="pt")
        input_ids = encodings.input_ids.to(device)

        with torch.no_grad():
            outputs = ppl_model(input_ids, labels=input_ids)
            neg_log_likelihood = outputs.loss
            nlls.append(neg_log_likelihood)

    if not nlls:
        return 0.0

    avg_ppl = torch.exp(torch.stack(nlls).mean()).item()
    return avg_ppl

perplexity_score = compute_perplexity(predictions)

# -------------------------------------------------------------
# 4. Diversity (Distinct-1 and Distinct-2 N-grams)
# -------------------------------------------------------------
def compute_distinct_n(texts, n=1):
    total_ngrams = 0
    unique_ngrams = set()

    for text in texts:
        tokens = text.lower().split()
        if len(tokens) < n:
            continue
        ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
        total_ngrams += len(ngrams)
        unique_ngrams.update(ngrams)

    if total_ngrams == 0:
        return 0.0

    return len(unique_ngrams) / total_ngrams

distinct_1 = compute_distinct_n(predictions, n=1)
distinct_2 = compute_distinct_n(predictions, n=2)

# -------------------------------------------------------------
# Display Benchmark Results
# -------------------------------------------------------------
print("\n--- Additional Evaluation Metrics ---")
print(f"METEOR Score: {meteor_results['meteor']:.4f}")
print(f"Average Token F1-Score: {avg_f1_score:.4f}")
print(f"Perplexity (GPT-2): {perplexity_score:.2f}")
print(f"Distinct-1 (Unigram Diversity): {distinct_1:.4f}")
print(f"Distinct-2 (Bigram Diversity): {distinct_2:.4f}")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



--- Additional Evaluation Metrics ---
METEOR Score: 0.3742
Average Token F1-Score: 0.3675
Perplexity (GPT-2): 71.89
Distinct-1 (Unigram Diversity): 0.3097
Distinct-2 (Bigram Diversity): 0.6333


##**IWF Detection Module**

In [15]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

class IWFDetector:
    def __init__(self):
        # Common vague frequency terms that weaken question discriminability
        self.vague_terms = {
            "usually", "often", "frequently", "sometimes",
            "generally", "rarely", "seldom", "occasionally"
        }
        # Absolute terms that often serve as test-taking cues
        self.absolute_terms = {
            "always", "never", "all", "none", "must", "completely", "entirely"
        }

    def check_stem_clues(self, stem, options):
        """Checks if the correct answer or key words are leaked in the question stem."""
        flaws = []
        stem_lower = stem.lower()
        for opt in options:
            if opt.lower() in stem_lower and len(opt.strip()) > 3:
                flaws.append(f"Answer Leaked in Stem: Option '{opt}' appears verbatim in question.")
        return flaws

    def check_all_or_none_above(self, options):
        """Checks for 'All of the above' or 'None of the above' options."""
        flaws = []
        patterns = [
            r"\ball of the above\b",
            r"\bnone of the above\b",
            r"\bboth [a-d] and [a-d]\b",
            r"\ball of these\b"
        ]
        for opt in options:
            for pat in patterns:
                if re.search(pat, opt, re.IGNORECASE):
                    flaws.append(f"Inclusive/Exclusive Option Flaw: Detected '{opt}'.")
        return flaws

    def check_vague_or_absolute_terms(self, stem, options):
        """Checks for vague qualifiers and absolute words."""
        flaws = []
        all_text = " ".join([stem] + options).lower().split()
        for word in all_text:
            cleaned = re.sub(r'[^\w\s]', '', word)
            if cleaned in self.vague_terms:
                flaws.append(f"Vague Frequency Term: Detected '{cleaned}'.")
            elif cleaned in self.absolute_terms:
                flaws.append(f"Absolute Word Cue: Detected '{cleaned}'.")
        return flaws

    def check_grammatical_agreement(self, stem, options):
        """Checks for grammatical cueing (e.g., stem ending with 'a' vs option starting with vowel)."""
        flaws = []
        stem_stripped = stem.strip().rstrip("?:_ ")
        last_word = stem_stripped.split()[-1].lower() if stem_stripped.split() else ""

        if last_word in ["a", "an"]:
            for opt in options:
                first_word = opt.strip().split()[0].lower() if opt.strip().split() else ""
                if first_word:
                    first_char = first_word[0]
                    if last_word == "a" and first_char in "aeiou":
                        flaws.append(f"Grammar Cue: Stem ends with 'a' but option '{opt}' starts with a vowel.")
                    elif last_word == "an" and first_char not in "aeiou":
                        flaws.append(f"Grammar Cue: Stem ends with 'an' but option '{opt}' starts with a consonant.")
        return flaws

    def check_length_outliers(self, correct_answer, distractors):
        """Flags if the correct answer is significantly longer/shorter than distractors."""
        flaws = []
        all_options = [correct_answer] + distractors
        lengths = [len(opt.split()) for opt in all_options]
        avg_len = sum(lengths) / len(lengths)
        ans_len = len(correct_answer.split())

        # Flag if correct answer is more than double the average length
        if ans_len > 2 * avg_len and ans_len > 4:
            flaws.append("Length Cue: Correct answer is significantly longer than distractors.")
        return flaws

    def evaluate_mcq(self, question_stem, correct_answer, distractors):
        """Runs the IWF suite and aggregates detected flaws."""
        all_options = [correct_answer] + distractors
        detected_flaws = []

        detected_flaws.extend(self.check_all_or_none_above(all_options))
        detected_flaws.extend(self.check_stem_clues(question_stem, [correct_answer]))
        detected_flaws.extend(self.check_vague_or_absolute_terms(question_stem, all_options))
        detected_flaws.extend(self.check_grammatical_agreement(question_stem, all_options))
        detected_flaws.extend(self.check_length_outliers(correct_answer, distractors))

        return {
            "is_valid": len(detected_flaws) == 0,
            "flaw_count": len(detected_flaws),
            "flaws": list(set(detected_flaws))
        }

In [16]:
detector = IWFDetector()

# Example 1: MCQ with multiple IWFs (Length cue, "All of the above", Absolute term)
test_item_1 = {
    "stem": "Which of the following always occurs during cellular respiration?",
    "correct_answer": "Conversion of biochemical energy from nutrients into adenosine triphosphate (ATP) in the mitochondria",
    "distractors": [
        "Production of glucose",
        "Absorption of sunlight",
        "All of the above"
    ]
}

# Example 2: Clean MCQ
test_item_2 = {
    "stem": "What algorithm is used to avoid deadlocks?",
    "correct_answer": "Banker's algorithm",
    "distractors": [
        "Round Robin algorithm",
        "Dijkstra's shortest path algorithm",
        "First-Come First-Served algorithm"
    ]
}

print("--- Test Item 1 Evaluation ---")
res1 = detector.evaluate_mcq(test_item_1["stem"], test_item_1["correct_answer"], test_item_1["distractors"])
print(f"Valid: {res1['is_valid']}")
for flaw in res1['flaws']:
    print(f" - {flaw}")

print("\n--- Test Item 2 Evaluation ---")
res2 = detector.evaluate_mcq(test_item_2["stem"], test_item_2["correct_answer"], test_item_2["distractors"])
print(f"Valid: {res2['is_valid']}")
print(f"Flaws: {res2['flaws']}")

--- Test Item 1 Evaluation ---
Valid: False
 - Length Cue: Correct answer is significantly longer than distractors.
 - Absolute Word Cue: Detected 'always'.
 - Absolute Word Cue: Detected 'all'.
 - Inclusive/Exclusive Option Flaw: Detected 'All of the above'.

--- Test Item 2 Evaluation ---
Valid: True
Flaws: []
